In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, GRU, SimpleRNN
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import time
np.random.seed(1234)
tf.random.set_seed(1234)

In [ ]:
sequence_length = 50
nb_features = 10
nb_out = 1

seq_array = np.random.rand(1000, sequence_length, nb_features)
label_array = np.random.randint(0, 2, size=(1000, nb_out))

In [ ]:
def build_model(model_type="LSTM"):
    model = Sequential()
    if model_type == "LSTM":
        model.add(LSTM(100, input_shape=(sequence_length, nb_features), return_sequences=True))
        model.add(Dropout(0.2))
        model.add(LSTM(50, return_sequences=False))
    elif model_type == "GRU":
        model.add(GRU(100, input_shape=(sequence_length, nb_features), return_sequences=True))
        model.add(Dropout(0.2))
        model.add(GRU(50, return_sequences=False))
    elif model_type == "RNN":
        model.add(SimpleRNN(100, input_shape=(sequence_length, nb_features), return_sequences=True))
        model.add(Dropout(0.2))
        model.add(SimpleRNN(50, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(nb_out, activation="sigmoid"))
    model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model


In [ ]:
def train_and_evaluate(model_type):
    model = build_model(model_type)
    start = time.time()
    history = model.fit(
        seq_array, label_array,
        epochs=10, batch_size=200,
        validation_split=0.05, verbose=1,
        callbacks=[EarlyStopping(monitor='val_loss', patience=1)]
    )
    end = time.time()
    print(f"{model_type} — время обучения: {end - start:.2f} сек")
    return model, history


In [ ]:
models = {}
histories = {}
for net in ["RNN", "LSTM", "GRU"]:
    models[net], histories[net] = train_and_evaluate(net)


In [ ]:
for name, model in models.items():
    print(f"\n{name} summary:")
    model.summary()


In [ ]:
def plot_history(history, title):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Val Loss")
    plt.title(f"{title} — Loss")
    plt.legend()
    plt.subplot(1,2,2)
    plt.plot(history.history["accuracy"], label="Train Acc")
    plt.plot(history.history["val_accuracy"], label="Val Acc")
    plt.title(f"{title} — Accuracy")
    plt.legend()
    plt.show()

for net, hist in histories.items():
    plot_history(hist, net)


In [ ]:
results = []
for net, hist in histories.items():
    val_acc = hist.history["val_accuracy"][-1]
    val_loss = hist.history["val_loss"][-1]
    results.append((net, val_acc, val_loss))
df_results = pd.DataFrame(results, columns=["Модель", "Val Accuracy", "Val Loss"])
df_results


In [ ]:
import tensorflow as tf
print("GPU доступен:" , tf.config.list_physical_devices('GPU'))
